**Step 1: Understand the Basics**

Q1. MapReduce vs. Apache Spark

MapReduce: This is an older Big Data tool. It is slow because it reads data from the physical hard drive, processes a tiny bit, and writes it back to the hard drive over and over again.

Apache Spark is much faster (up to 100x faster!) because of in-memory processing. Instead of saving to the hard drive, it loads the data directly into the computer's RAM. It also uses DataFrames, which are immutable (meaning once created, they cannot be accidentally changed, only transformed into new DataFrames).


Q2. Spark DataFrames

Spark organizes data into DataFrames (which look like standard tables with rows and columns).

Immutability- Once a DataFrame is created, it cannot be changed. If you want to clean data, Spark creates a brand new DataFrame, keeping your original raw data safe.


Step 2. Installing Pyspark and Libraries

In [3]:
# 1. Install PySpark (Required for Google Colab)
!pip install pyspark

# 2. Import the required Spark libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, min, max, count

# 3. Create the Spark session
spark = SparkSession.builder \
    .appName("Superstore_Data_Processing") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


STep 3. Loading the data

In [10]:
file_path = "/content/Sample - Superstore.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True, escape='"')

print("First 5 Rows")
df.show(5)

First 5 Rows
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|B

In [11]:
print("Data Schema")
df.printSchema()

Data Schema
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Step 4. Data cleaning

In [12]:
# 1. Remove duplicate rows
cleaned_df = df.dropDuplicates()

# 2. dropping rows that contains null value
cleaned_df = cleaned_df.dropna()

# 3. count the rows to see if anything was removed
original_count = df.count()
cleaned_count = cleaned_df.count()

print(f"Original Row Count: {original_count}")
print(f"Cleaned Row Count: {cleaned_count}")
print(f"Total Rows Cleaned/Removed: {original_count - cleaned_count}")

Original Row Count: 9994
Cleaned Row Count: 9994
Total Rows Cleaned/Removed: 0


Step 5. Filter Data

In [13]:
from pyspark.sql.functions import col


cleaned_df = cleaned_df.withColumn("Sales", col("Sales").cast("double"))

filtered_df = cleaned_df \
    .filter(col("Category") == "Furniture") \
    .filter(col("Region") == "West") \
    .filter(col("Sales") >= 100)

# final results
print("Filtered Data")
filtered_df.show(5)

print(f"Total Rows matching our filters: {filtered_df.count()}")

Filtered Data
+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+-------------+----------+-----------+------+---------------+---------+------------+--------------------+-------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|  Segment|      Country|         City|     State|Postal Code|Region|     Product ID| Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+---------+-------------+-------------+----------+-----------+------+---------------+---------+------------+--------------------+-------+--------+--------+---------+
|   416|CA-2017-142636| 11/3/2017| 11/7/2017|Standard Class|   KC-16675|  Kimberly Carter|Corporate|United States|      Seattle|Washington|      98105|  West|FUR-CH-10001891|Furniture|      Chairs|Global Deluxe Off...|307

Step 7 & Step 8: Aggregation and Grouping

In [14]:

from pyspark.sql.functions import sum, avg, min, max, count

aggregated_df = cleaned_df.groupBy("Region").agg(
    count("*").alias("Total_Orders"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sale_Value"),
    max("Sales").alias("Highest_Single_Sale"),
    min("Sales").alias("Lowest_Single_Sale")
)

# results
print("Sales Aggregation by Region ")
aggregated_df.show()

Sales Aggregation by Region 
+-------+------------+-----------------+------------------+-------------------+------------------+
| Region|Total_Orders|      Total_Sales|Average_Sale_Value|Highest_Single_Sale|Lowest_Single_Sale|
+-------+------------+-----------------+------------------+-------------------+------------------+
|  South|        1620|       391721.905|241.80364506172842|           22638.48|             1.167|
|Central|        2323|501239.8907999995|215.77266069737388|           17499.95|             0.444|
|   East|        2848|678781.2400000005|238.33610955056196|          11199.968|             0.852|
|   West|        3203|725457.8245000001|226.49323275054638|           13999.96|              0.99|
+-------+------------+-----------------+------------------+-------------------+------------------+



Wide Transformations: When we used groupBy("Region") in the previous step, Spark couldn't just look at one row at a time. It had to look across all its different partitions (chunks of data) to gather all the "West" rows together. This is called a "Wide Transformation."

Shuffling: To actually group those regions together, Spark has to physically move data around between its different processors over the network. This physical data movement is called a Shuffle, and it is usually the slowest and most resource-heavy part of a Spark job.

In [15]:
output_path = "/content/output_results"
aggregated_df.write.csv(output_path, header=True, mode="overwrite")

print("results saved.")

Success! Full data pipeline complete and results saved.
